# RQ3 results figures -- plan

Builds the figures backing RQ3's plot-specific curve-deviation claims in
`documentation/august_draft/5_Chapter_results_evaluation/f_written_draft_v1_18thaug.tex`.

**Figures in this notebook**

| Figure | Status | Question it answers |
|---|---|---|
| R3-1 | **Must-have** | Does GNNWR's accuracy edge over EN/XGBoost correspond to it actually reducing leftover spatial structure, and does that hold on both cohorts? |
| R3-2 | **Must-have** | Where do growth-curve deviations occur, and does the spatial pattern line up with the proposed reasons (boundary proximity, compartment archetype)? |
| R3-3 | Optional (build if time allows) | Do EN, XGBoost, and GNNWR agree on which variable matters most, on both cohorts? |

**Not built here (cut for time)**: R3-4 (compartment-archetype representative trajectories, small
multiples) was considered and dropped -- illustrative only, and R3-2's Panel B already shows
archetypes spatially; its own plotting code did not exist yet in this project either.

**Data-source honesty note**: R3-1 Panel A (EN/XGBoost/GNNWR R2 by set) and R3-3 (rank agreement)
are transcribed from `TEMP_results/TEMP_rq3_en_xgb_results_2026-08-11.tex` and
`TEMP_results/TEMP_rq3_gnnwr_results_2026-08-11.tex` with citation, since this session did not
independently verify every underlying per-set predictions.csv path for RQ3's EN/XGBoost fits.
**Panel B is now FINAL, not placeholder** -- transcribed exactly from the published
`tab:results-rq3-morans` table in the results draft (2026-08-19), itself the corrected
semivariogram-informed Moran's I re-run. R3-1 Panel C, R3-2 (all panels), and the GNNWR pooling
all use live, verified file reads.

**Style / uncertainty convention**: see `notebooks/results_figures_style.py`. R3-1 Panel A uses
the same **1,000-resample** bootstrap CI convention as everywhere else in this project. Panel B
(Moran's I) has no CI. R3-2 has no uncertainty anywhere -- deterministic target values and raw
observations.

In [ ]:
# Purpose: make the models/ package (and notebooks/results_figures_style.py) importable from
# this notebook. Same convention already used across this project's other notebooks (e.g.
# notebooks/model_results/baseline_results.ipynb): walk upward until a folder containing both
# README.md and data/ is found -- that is the project root.

import sys
from pathlib import Path

notebook_directory = Path.cwd().resolve()
project_root = next(
    folder for folder in [notebook_directory, *notebook_directory.parents]
    if (folder / "README.md").exists() and (folder / "data").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("project_root:", project_root)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from models.common.geo import load_compartment_boundaries, load_plot_coordinates
from notebooks.results_figures_style import (
    COLOR_EN, COLOR_XGBOOST, COLOR_GNNWR, COLOR_NEUTRAL_EDGE, DIVERGING_CMAP, apply_rcparams,
)

apply_rcparams()

FIGURES_DIR = project_root / "figures" / "fig_results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

GNNWR_DIR = project_root / "outputs" / "growth_curve_attribution" / "gnnwr"
N_FOLDS = 5
SETS = ["Set2", "Set3", "Set4"]

## Figure R3-1 -- GNNWR's accuracy edge and its spatial-structure meaning

**Research question**: RQ3 items 2-3 -- does GNNWR's R2 edge over EN/XGBoost on 4survey
correspond to an independent reduction in residual Moran's I, and does that correspondence hold
or break down on 6survey?

**Data**: Panels A/B transcribed (with citation) from `TEMP_rq3_en_xgb_results_2026-08-11.tex` /
`TEMP_rq3_gnnwr_results_2026-08-11.tex`. Panel C -- pools the 5 fold
`gnnwr_nested_set4_gated_all_vif_4survey_reffull_fold{0-4}of5_test_predictions.csv` files' own
`coef_CanopyCover`, `x`, `y`, `cpmt` columns into one per-plot value -- live read, no new fitting.

**Encoding**: Panel A -- 4survey only (R2, one point+CI per model per set); Panel B -- both
cohorts (Moran's I by set, one line per model); Panel C -- diverging map of GNNWR's own local
CanopyCover coefficient, compartment mean, centred at 0.

In [ ]:
# Transcribed from TEMP_rq3_en_xgb_results_2026-08-11.tex and TEMP_rq3_gnnwr_results_2026-08-11.tex
# -- see the markdown cell above for why these are hardcoded rather than re-read live.
PANEL_A_4SURVEY = {  # {set: {model: (r2, ci_low, ci_high)}}
    "Set2": {"Elastic Net": (0.286, 0.241, 0.325), "XGBoost": (0.266, 0.214, 0.308), "GNNWR": (0.320, 0.270, 0.369)},
    "Set3": {"Elastic Net": (0.274, 0.223, 0.316), "XGBoost": (0.281, 0.235, 0.320), "GNNWR": (0.319, 0.263, 0.365)},
    "Set4": {"Elastic Net": (0.240, 0.192, 0.286), "XGBoost": (0.250, 0.201, 0.295), "GNNWR": (0.294, 0.236, 0.347)},
}

# FINAL 2026-08-19 -- exact values, transcribed directly from the published
# tab:results-rq3-morans table in f_written_draft_v1_18thaug.tex (lines 217-222), not an estimate
# or a scaled placeholder. "no structure" (semivariogram flat even at the 5,000m search window,
# genuine null) is represented as None -- plotted as explicitly missing, never interpolated. The
# 6survey values that DID resolve only did so after widening the search window to 10,000m
# (exceeds_window status, not resolved) and are not significant (p=0.05-0.31) -- real numbers,
# but provisional, not equivalent evidence to the resolved 4survey column (see the table's own
# caption). Every 4survey cell is resolved and significant (p=0.001).
PANEL_B_MORANS_I = {  # {cohort: {set: {model: morans_i}}}
    "4survey": {
        "Set2": {"Elastic Net": 0.147, "XGBoost": 0.148, "GNNWR": 0.100},
        "Set3": {"Elastic Net": 0.167, "XGBoost": 0.144, "GNNWR": 0.109},
        "Set4": {"Elastic Net": 0.145, "XGBoost": 0.107, "GNNWR": 0.082},
    },
    "6survey": {  # None = no structure (unresolved even at 10,000m) -- plotted as explicitly missing
        "Set2": {"Elastic Net": None, "XGBoost": -0.0002, "GNNWR": -0.0002},  # XGB/GNNWR: exceeds_window, ns
        "Set3": {"Elastic Net": None, "XGBoost": None, "GNNWR": None},
        "Set4": {"Elastic Net": None, "XGBoost": -0.0002, "GNNWR": -0.0002},  # XGB/GNNWR: exceeds_window, ns
    },
}

In [ ]:
fig, (ax_a, ax_b) = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
model_colors_r3 = {"Elastic Net": COLOR_EN, "XGBoost": COLOR_XGBOOST, "GNNWR": COLOR_GNNWR}

for model, color in model_colors_r3.items():
    r2_vals = [PANEL_A_4SURVEY[s][model][0] for s in SETS]
    lows = [PANEL_A_4SURVEY[s][model][1] for s in SETS]
    highs = [PANEL_A_4SURVEY[s][model][2] for s in SETS]
    yerr = [[r - lo for r, lo in zip(r2_vals, lows)], [hi - r for r, hi in zip(r2_vals, highs)]]
    ax_a.errorbar(SETS, r2_vals, yerr=yerr, marker="o", color=color, label=model, capsize=3, linewidth=1.8)
ax_a.set_ylabel("Test $R^2$ (95% CI, 1,000 resamples)\n4survey only")
ax_a.set_title("Panel A: accuracy (4survey only -- 6survey R2 is not a stable single value)", fontsize=10)
ax_a.legend(fontsize=8, frameon=False)

for cohort, linestyle in [("4survey", "-"), ("6survey", "--")]:
    for model, color in model_colors_r3.items():
        # None = semivariogram range did not resolve (no measurable spatial structure at this
        # cohort/set) -- plotted as an explicit gap (NaN breaks a matplotlib line automatically),
        # never silently interpolated or dropped, per the caveat attached to PANEL_B_MORANS_I above.
        raw_values = [PANEL_B_MORANS_I[cohort][s][model] for s in SETS]
        values = [v if v is not None else float("nan") for v in raw_values]
        ax_b.plot(SETS, values, marker="o", color=color, linestyle=linestyle,
                  label=f"{model} ({cohort})", linewidth=1.6)
        for set_index, (set_name, raw_value) in enumerate(zip(SETS, raw_values)):
            if raw_value is None:
                ax_b.scatter([set_index], [0], marker="x", color=color, s=25, zorder=3)
ax_b.annotate("x = semivariogram range did not resolve (no detectable spatial structure)",
              xy=(0, 0), xytext=(0.02, 0.02), textcoords="axes fraction", fontsize=6.5, color="#555555")
ax_b.set_ylabel("Residual Moran's I")
ax_b.set_title("Panel B: spatial structure, both cohorts (solid=4survey, dashed=6survey)", fontsize=10)
ax_b.legend(fontsize=7, frameon=False, ncol=2)
ax_b.set_xlabel("Feature set")

fig.suptitle("GNNWR's accuracy edge and its spatial-structure meaning", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Panel C: pool GNNWR's own local CanopyCover coefficient across the 5 folds, one value per plot ---
gnnwr_frames = []
for fold in range(N_FOLDS):
    p = GNNWR_DIR / f"gnnwr_nested_set4_gated_all_vif_4survey_reffull_fold{fold}of5_test_predictions.csv"
    fold_df = pd.read_csv(p, usecols=["identification", "cpmt", "x", "y", "coef_CanopyCover"])
    gnnwr_frames.append(fold_df)
gnnwr_pooled = pd.concat(gnnwr_frames, ignore_index=True).drop_duplicates(subset="identification")
print(f"{len(gnnwr_pooled):,} plots with a pooled local CanopyCover coefficient")

compartment_mean_coef = gnnwr_pooled.groupby("cpmt")["coef_CanopyCover"].mean().reset_index()
boundaries = load_compartment_boundaries()
mapped = boundaries.merge(compartment_mean_coef, on="cpmt", how="left")

fig, ax_c = plt.subplots(figsize=(7, 6))
vmax = mapped["coef_CanopyCover"].abs().max()
norm = mpl.colors.TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)
mapped.plot(column="coef_CanopyCover", cmap=DIVERGING_CMAP, norm=norm, ax=ax_c,
            edgecolor="white", linewidth=0.2, missing_kwds={"color": "#DDDDDD"})
sm = plt.cm.ScalarMappable(cmap=DIVERGING_CMAP, norm=norm)
fig.colorbar(sm, ax=ax_c, shrink=0.8, label="Local coef_CanopyCover (compartment mean)")
ax_c.set_title("Panel C: GNNWR's own local CanopyCover coefficient (Set4, 4survey)", fontsize=10)
ax_c.set_aspect("equal")
ax_c.set_xticks([]); ax_c.set_yticks([])
plt.show()

## Figure R3-2 -- where growth-curve deviations occur, and what explains them

**Research question**: RQ3 items 4-6, 9 -- does the spatial pattern of `local_y_max_difference`
line up with boundary proximity and compartment archetype, and are the recurring worst-residual
plots visibly the same points the map and the trajectories are both talking about?

**Data**: Panel A -- `build_plot_level_table("4survey")` for the deviation map, PLUS
`TEMP_results/rq3_outlier_disturbance_crossref.csv` for the real, cross-model-derived recurring
outlier population (NOT a proxy threshold rule -- this is the actual project file behind the
"roughly ten plots consistently account for the worst residuals" claim, draft line 235). Filtered
to 4survey: exactly 8 plots, in 6 compartments, splitting cleanly by `residual_range` into 4
"smooth" (<6m) and 4 "unstable" (13-15m) -- matching the draft's own "half and half" description.
Panel B -- `rq3_compartment_archetype_check.py`'s `build_plot_table()` + `classify_compartments()`.
Both panels get boundary lines for the 6 flagged compartments only (not all 231 -- avoids clutter,
and these are exactly the compartments the outlier discussion is about). Panel C -- 2 smooth + 2
unstable examples selected FROM this real 8-plot population (most extreme within each group), each
labelled A/B/C/D, matching the same letters marked on Panel A.

In [ ]:
from models.growth_curve_attribution.scale_comparison_check import build_plot_level_table
from models.growth_curve_attribution.rq3_compartment_archetype_check import build_plot_table, classify_compartments
from models.growth_curve_attribution.data import load_filtered_growth_curve_table

panel_a_table = build_plot_level_table("4survey")
print(f"Panel A: {len(panel_a_table):,} plots, local_y_max_difference range "
      f"[{panel_a_table['local_y_max_difference'].min():.1f}, {panel_a_table['local_y_max_difference'].max():.1f}] m")

panel_b_plot_table, _ = build_plot_table("4survey")
compartment_archetypes, _ = classify_compartments(panel_b_plot_table)
print(f"Panel B: {len(compartment_archetypes)} compartments classified")
print(compartment_archetypes["pattern"].value_counts())

# --- Real, cross-model-derived recurring outlier population (not a proxy threshold rule) ---
outlier_crossref = pd.read_csv(project_root / "TEMP_results" / "rq3_outlier_disturbance_crossref.csv")
outliers_4survey = outlier_crossref[outlier_crossref["cohort"] == "4survey"].copy()
outliers_4survey["sub_population"] = np.where(outliers_4survey["residual_range"] < 6, "smooth", "unstable")
# Join x/y from panel_a_table (already has one row per plot with coordinates) -- no new lookup needed.
outliers_4survey = outliers_4survey.merge(
    panel_a_table[["identification", "x", "y"]], on="identification", how="left",
)
flagged_compartments = sorted(outliers_4survey["cpmt"].unique().tolist())
print(f"Real recurring-outlier population (4survey): {len(outliers_4survey)} plots, "
      f"{outliers_4survey['sub_population'].value_counts().to_dict()}, "
      f"in {len(flagged_compartments)} compartments: {flagged_compartments}")

# 2 smooth + 2 unstable examples (most extreme within each group) -- computed here so both the
# map (this section) and the trajectory panel (below) mark/label the exact same 4 plots.
smooth_examples = outliers_4survey[outliers_4survey["sub_population"] == "smooth"].sort_values(
    "residual_range").head(2)
unstable_examples = outliers_4survey[outliers_4survey["sub_population"] == "unstable"].sort_values(
    "residual_range", ascending=False).head(2)
selected_plots = list(smooth_examples["identification"]) + list(unstable_examples["identification"])
selected_labels = ["smooth", "smooth", "unstable", "unstable"]
panel_letters = ["A", "B", "C", "D"]
selected_xy = pd.concat([smooth_examples, unstable_examples])[["identification", "x", "y"]].reset_index(drop=True)
print("Selected plots for Panel C (labelled to match Panel A's markers):",
      dict(zip(panel_letters, selected_plots)))

In [ ]:
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(13, 6.5), sharex=False, sharey=False)

# --- Panel A: deviation map, compartment-mean local_y_max_difference ---
compartment_mean_deviation = panel_a_table.groupby("cpmt")["local_y_max_difference"].mean().reset_index()
boundaries = load_compartment_boundaries()
mapped_a = boundaries.merge(compartment_mean_deviation, on="cpmt", how="left")
vmax = mapped_a["local_y_max_difference"].abs().quantile(0.98)  # robust to a handful of extreme outlier compartments
norm = mpl.colors.TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)
mapped_a.plot(column="local_y_max_difference", cmap=DIVERGING_CMAP, norm=norm, ax=ax_a,
              edgecolor="white", linewidth=0.2, missing_kwds={"color": "#DDDDDD"})
sm = plt.cm.ScalarMappable(cmap=DIVERGING_CMAP, norm=norm)
fig.colorbar(sm, ax=ax_a, shrink=0.75, label="Mean local_y_max_difference (m)")
ax_a.set_title("Panel A: deviation from yield-class benchmark", fontsize=10)
ax_a.set_aspect("equal"); ax_a.set_xticks([]); ax_a.set_yticks([])

# --- Panel B: compartment archetype map, qualitative palette (deliberately different colour system) ---
mapped_b = boundaries.merge(compartment_archetypes[["cpmt", "pattern"]], on="cpmt", how="left")
mapped_b.plot(column="pattern", categorical=True, legend=True, ax=ax_b,
              edgecolor="white", linewidth=0.2, cmap="Set2",
              missing_kwds={"color": "#DDDDDD"},
              legend_kwds={"fontsize": 7, "loc": "lower left", "frameon": True})
ax_b.set_title("Panel B: compartment archetype", fontsize=10)
ax_b.set_aspect("equal"); ax_b.set_xticks([]); ax_b.set_yticks([])

# --- Boundary lines for the 6 flagged compartments only (not all 231 -- these are exactly the
# compartments the outlier discussion is about, so no clutter risk) ---
flagged_boundary_polygons = boundaries[boundaries["cpmt"].isin(flagged_compartments)]
for ax in (ax_a, ax_b):
    flagged_boundary_polygons.boundary.plot(ax=ax, color="black", linewidth=1.0, zorder=4)

# --- Outlier plot markers: star=smooth, triangle=unstable, on BOTH panels so the same points are
# visible against both the deviation colour and the archetype colour ---
marker_by_subpop = {"smooth": "*", "unstable": "^"}
for ax in (ax_a, ax_b):
    for subpop, marker in marker_by_subpop.items():
        subset = outliers_4survey[outliers_4survey["sub_population"] == subpop]
        ax.scatter(subset["x"], subset["y"], marker=marker, s=90, facecolor="none",
                   edgecolor="black", linewidth=1.3, zorder=5)
    # Letter labels ONLY for the 4 plots Panel C actually shows -- ties the map directly to the
    # trajectory examples below, not just to the sub-population as a whole.
    for _, row in selected_xy.iterrows():
        letter = panel_letters[list(selected_plots).index(row["identification"])]
        ax.annotate(letter, xy=(row["x"], row["y"]), xytext=(5, 5), textcoords="offset points",
                    fontsize=9, fontweight="bold", zorder=6)

fig.suptitle("Where growth-curve deviations occur, and what explains them\n"
             "(black outlines mark the real recurring-outlier plots: * smooth, ^ unstable; "
             "letters A-D match Panel C below; black lines mark their 6 compartments)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- Panel C: the same 4 plots (2 smooth + 2 unstable) already selected and marked A-D on the
# map above -- selected_plots/selected_labels/panel_letters computed there, reused here unchanged
# so the map and these trajectories are guaranteed to refer to the exact same plots. ---
growth_rows = load_filtered_growth_curve_table("4survey")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=False)
for ax, plot_id, label, letter in zip(axes, selected_plots, selected_labels, panel_letters):
    plot_rows = growth_rows[growth_rows["identification"] == plot_id].sort_values("Age")
    ax.plot(plot_rows["Age"], plot_rows["elev_percentile_95th"], marker="o", color=COLOR_NEUTRAL_EDGE,
            linewidth=1.6, label="observed")
    ax.plot(plot_rows["Age"], plot_rows["top_height95_yldc_predicted"], linestyle="--", color=COLOR_XGBOOST,
            linewidth=1.4, label="yield-class benchmark")
    ax.set_title(f"{letter} -- {label}, plot {int(plot_id)}", fontsize=8)
    ax.set_xlabel("Age (years)", fontsize=8)
axes[0].set_ylabel("Top height (m)")
axes[0].legend(fontsize=7, frameon=False)
fig.suptitle("Panel C: representative trajectories vs. yield-class benchmark "
             "(letters match Panel A's marked points)", fontsize=11)
plt.tight_layout()
plt.show()

## Figure R3-3 (optional) -- does CanopyCover's rank agree across all three attribution models?

**Research question**: RQ3 item 1 -- do EN, XGBoost, and GNNWR agree on which variable is most
important, separately for 4survey and 6survey?

**Data source note**: transcribed (with citation) from `TEMP_rq3_en_xgb_results_2026-08-11.tex`
(EN coefficient rank, XGBoost gain-importance rank) and `TEMP_rq3_gnnwr_local_coef_rank_2026-08-16.tex`
(GNNWR local-coefficient rank) -- this session did not independently re-verify every underlying
gain-importance/rank computation, only the headline conclusion (CanopyCover #1 on 4survey by all
three methods; on 6survey EN/GNNWR still rank it #1, XGBoost does not). **Replace the placeholder
ranks below with the real per-set ranks from those two source files before using this figure.**

**Encoding**: slopegraph, one panel per cohort, three columns (EN, XGBoost, GNNWR), y = rank (1 at
top). CanopyCover's own line in a bold colour; every other variable in muted grey.

In [ ]:
# PLACEHOLDER ranks -- confirm against TEMP_rq3_en_xgb_results_2026-08-11.tex and
# TEMP_rq3_gnnwr_local_coef_rank_2026-08-16.tex before using this figure. Only CanopyCover's own
# rank is asserted with confidence per the source text; the other variables' exact ranks/names
# here are illustrative placeholders standing in for "whatever displaces CanopyCover."
RANKS = {
    "4survey": {"Elastic Net": {"CanopyCover": 1, "time_since_thinning": 2, "slope_degrees": 3},
                "XGBoost": {"CanopyCover": 1, "time_since_thinning": 2, "slope_degrees": 3},
                "GNNWR": {"CanopyCover": 1, "time_since_thinning": 2, "slope_degrees": 3}},
    "6survey": {"Elastic Net": {"CanopyCover": 1, "cpmt_compactness_ratio": 2, "time_since_thinning": 3},
                "XGBoost": {"cpmt_compactness_ratio": 1, "CanopyCover": 2, "time_since_thinning": 3},
                "GNNWR": {"CanopyCover": 1, "cpmt_compactness_ratio": 2, "time_since_thinning": 3}},
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
methods = ["Elastic Net", "XGBoost", "GNNWR"]
for ax, cohort in zip(axes, ["4survey", "6survey"]):
    variables = set()
    for method_ranks in RANKS[cohort].values():
        variables.update(method_ranks.keys())
    for variable in variables:
        ys = [RANKS[cohort][m].get(variable, len(variables) + 1) for m in methods]
        color = COLOR_XGBOOST if variable == "CanopyCover" else "#BBBBBB"
        linewidth = 2.4 if variable == "CanopyCover" else 1.0
        ax.plot(range(3), ys, marker="o", color=color, linewidth=linewidth, zorder=3 if variable == "CanopyCover" else 1)
        ax.text(-0.1, ys[0], variable, ha="right", va="center", fontsize=7)
    ax.set_xticks(range(3))
    ax.set_xticklabels(methods, fontsize=9)
    ax.invert_yaxis()
    ax.set_ylabel("Rank (1 = most important)")
    ax.set_title(cohort, fontsize=11)
    ax.set_xlim(-0.6, 2.2)

fig.suptitle("Does CanopyCover's rank agree across EN, XGBoost, and GNNWR? (PLACEHOLDER DATA)", fontsize=11)
plt.tight_layout()
plt.show()